# Test set

Inquiries and "golden" responses from rfq.xlsx; checks (golden codes found in the catalog, unlinked rows, discontinued products); division into development and evaluation groups

Provide: Test set Q, split

In [28]:
# libraries

import pandas as pd
import re
import numpy as np
import json
import os

## 1 Load data

In [2]:
rfq_raw = pd.read_excel("../data/rfq.xlsx")
catalog = pd.read_csv("../data/clean/catalog_clean.csv", dtype=str)

print(f"Loaded {len(rfq_raw)} RFQ rows.")
print(f"Loaded {len(catalog):,} catalog products.")

print("\nRFQ columns:", rfq_raw.columns.tolist())
rfq_raw.head(3)

Loaded 692 RFQ rows.
Loaded 74,525 catalog products.

RFQ columns: ['roq', 'request', 'requirements', 'article_matched', 'product_matched']


,roq,request,requirements,article_matched,product_matched
0,1,TURVAKENGÄT JA - NILKKURIT KESÄKÄYTTÖÖN\nTurva...,Käyttötarkoitus: sporttinen puolikenkämallinen...,M418156035918 1,TURVAJALKINE ROCK LOW 35
1,1,TURVAKENGÄT JA - NILKKURIT KESÄKÄYTTÖÖN\nTurva...,Käyttötarkoitus: sporttinen puolikenkämallinen...,M418156036918 1,TURVAJALKINE ROCK LOW 36
2,1,TURVAKENGÄT JA - NILKKURIT KESÄKÄYTTÖÖN\nTurva...,Käyttötarkoitus: sporttinen puolikenkämallinen...,M418156037918 1,TURVAJALKINE ROCK LOW 37


## 2 Request of quotation data cleaning

In [4]:
# same cleaning function as used for catalog data cleaning
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Lowercase
    text = text.lower()
    # Normalize dimensions: '8 x 40' -> '8x40'
    text = re.sub(r'(\d)\s*x\s*(\d)', r'\1x\2', text)
    # Strip 'mm' directly following a dimension: 'm8x40 mm' -> 'm8x40'
    text = re.sub(r'(x\d+(?:[.,]\d+)?)\s*mm\b', r'\1', text)
    # Normalize multiple spaces to a single space
    return ' '.join(text.split())

In [5]:
# Parse requirement strings into structured format

def parse_requirements(text):
    if not isinstance(text, str):
        return {}
    
    parts = [p.strip() for p in text.split('|') if p.strip()]
    parsed = {}
    general = []
    
    for p in parts:
        if ':' in p:
            k, v = p.split(':', 1)
            k, v = k.strip(), v.strip()
            # If key already exists (e.g. multiple 'Pakollinen'), store as list
            if k in parsed:
                if isinstance(parsed[k], list):
                    parsed[k].append(v)
                else:
                    parsed[k] = [parsed[k], v]
            else:
                parsed[k] = v
        else:
            general.append(p)
            
    if general:
        parsed['other'] = general
    return parsed

In [6]:
# Process requests and requirements
rfq_raw['clean_query'] = rfq_raw['request'].apply(clean_text)
rfq_raw['requirements_dict'] = rfq_raw['requirements'].apply(parse_requirements)
rfq_raw['requirements_json'] = rfq_raw['requirements_dict'].apply(lambda d: json.dumps(d, ensure_ascii=False))

In [7]:
# Preview a sample row (row 87: screw request)
sample_row = rfq_raw.iloc[87]
print("Query original: ", repr(sample_row['request']))
print("Query cleaned:  ", repr(sample_row['clean_query']))
print("\nParsed requirements:")
print(json.dumps(sample_row['requirements_dict'], ensure_ascii=False, indent=2))

Query original:  'Ruuvi (uppokantainen, torxpää, 3-5mm x 12-100mm) => 12mm'
Query cleaned:   'ruuvi (uppokantainen, torxpää, 3-5mm x 12-100mm) => 12mm'

Parsed requirements:
{
  "Tuote": "uppokantainen ruuvi",
  "Kanta": "torx",
  "Halkaisija": "3–5 mm",
  "Pituusvalikoima": "12–100 mm",
  "Rivillä pyydetty pituus": "12 mm",
  "other": [
    "Soveltuu ammattikäyttöön",
    "Hyvälaatuinen ja turvallinen",
    "CE-merkitty; poikkeuksena tuotteet, joille ei saa CE-merkintää ja jotka täyttävät rakennustuotteiden tuotehyväksyntää koskevan asetuksen 555/2013"
  ]
}


## 3 Match RFQ articles to catalog and extract the product_number

In [17]:
# ensure clean_query is populated
if 'clean_query' not in rfq_raw.columns:
    rfq_raw['clean_query'] = rfq_raw['request'].apply(clean_text)

# Build lookup maps from catalog: article_number -> product_number
cat_art_to_prod = dict(zip(catalog['article_number'].astype(str).str.strip(), catalog['product_number'].astype(str).str.strip()))
cat_art_ns_to_prod = dict(zip(catalog['article_number'].astype(str).str.replace(' ', ''), catalog['product_number'].astype(str).str.strip()))
cat_prod_to_prod = dict(zip(catalog['product_number'].astype(str).str.strip(), catalog['product_number'].astype(str).str.strip()))
cat_prod_ns_to_prod = dict(zip(catalog['product_number'].astype(str).str.replace(' ', ''), catalog['product_number'].astype(str).str.strip()))

In [18]:
def get_gold_product_number(raw_art):
    s = str(raw_art).strip()
    # exact match on article_number
    if s in cat_art_to_prod:
        return cat_art_to_prod[s]
    # match without whitespace (handles spacing variations)
    s_ns = s.replace(' ', '')
    if s_ns in cat_art_ns_to_prod:
        return cat_art_ns_to_prod[s_ns]
    # match against product_number directly
    if s in cat_prod_to_prod:
        return cat_prod_to_prod[s]
    if s_ns in cat_prod_ns_to_prod:
        return cat_prod_ns_to_prod[s_ns]
    # first token match
    tok = s.split()[0]
    if tok in cat_prod_to_prod:
        return cat_prod_to_prod[tok]
    # prefix match
    for ns_art, p in cat_art_ns_to_prod.items():
        if ns_art.startswith(s_ns) or s_ns.startswith(ns_art):
            return p
    return None

In [19]:
# Resolve canonical product_number from catalog
rfq_raw['gold_product_number'] = rfq_raw['article_matched'].apply(get_gold_product_number)
rfq_raw['catalog_found'] = rfq_raw['gold_product_number'].notna()

In [20]:
# Split into verified matched test set and unlinked/discontinued set
matched_rfq = rfq_raw[rfq_raw['catalog_found']].copy()
unlinked_rfq = rfq_raw[~rfq_raw['catalog_found']].copy()

In [21]:
# Attach catalog metadata (label and categories) using the matched product_number
catalog_meta = catalog.drop_duplicates(subset=['product_number'])[['product_number', 'product_label', 'clean_label', 'area_name', 'group_name', 'class_name']]
matched_rfq = matched_rfq.merge(catalog_meta, left_on='gold_product_number', right_on='product_number', how='left')

print(f"Total RFQ rows:          {len(rfq_raw)}")
print(f"Matched in catalog:      {len(matched_rfq)} ({len(matched_rfq)/len(rfq_raw):.1%})")
print(f"Unlinked / discontinued: {len(unlinked_rfq)} ({len(unlinked_rfq)/len(rfq_raw):.1%})")

Total RFQ rows:          692
Matched in catalog:      663 (95.8%)
Unlinked / discontinued: 29 (4.2%)


In [23]:
# Preview sample rows: see how article_matched links to canonical product_number
matched_rfq[['roq', 'article_matched', 'gold_product_number', 'product_matched', 'product_label', 'clean_query']].head(5)

,roq,article_matched,gold_product_number,product_matched,product_label,clean_query
0,1,M418156035918 1,M418156035,TURVAJALKINE ROCK LOW 35,TURVAJ ROCK LOW 35,turvakengät ja - nilkkurit kesäkäyttöön turvak...
1,1,M418156036918 1,M418156036,TURVAJALKINE ROCK LOW 36,TURVAJ ROCK LOW 36,turvakengät ja - nilkkurit kesäkäyttöön turvak...
2,1,M418156037918 1,M418156037,TURVAJALKINE ROCK LOW 37,TURVAJ ROCK LOW 37,turvakengät ja - nilkkurit kesäkäyttöön turvak...
3,1,M418156038918 1,M418156038,TURVAJALKINE ROCK LOW 38,TURVAJ ROCK LOW 38,turvakengät ja - nilkkurit kesäkäyttöön turvak...
4,1,M418156039918 1,M418156039,TURVAJALKINE ROCK LOW 39,TURVAJ ROCK LOW 39,turvakengät ja - nilkkurit kesäkäyttöön turvak...


## 4 Split test set in dev and eva sets

In [24]:
# ensure requirements_json exists
if 'requirements_json' not in matched_rfq.columns and 'requirements' in matched_rfq.columns:
    matched_rfq['requirements_dict'] = matched_rfq['requirements'].apply(parse_requirements)
    matched_rfq['requirements_json'] = matched_rfq['requirements_dict'].apply(lambda d: json.dumps(d, ensure_ascii=False))


# group split by unique queries to avoid query leakage between splits
np.random.seed(42)
unique_queries = matched_rfq['clean_query'].drop_duplicates().sample(frac=1.0, random_state=42).tolist()
split_point = int(len(unique_queries) * (1 / 3))
dev_queries = set(unique_queries[:split_point])
eval_queries = set(unique_queries[split_point:])
dev_set = matched_rfq[matched_rfq['clean_query'].isin(dev_queries)].copy()
eval_set = matched_rfq[matched_rfq['clean_query'].isin(eval_queries)].copy()
print(f"Total unique queries: {len(unique_queries)}")
print(f"  Development queries: {len(dev_queries)} ({len(dev_queries)/len(unique_queries):.1%}) | {len(dev_set)} rows")
print(f"  Evaluation queries:  {len(eval_queries)} ({len(eval_queries)/len(unique_queries):.1%}) | {len(eval_set)} rows")


Total unique queries: 408
  Development queries: 136 (33.3%) | 235 rows
  Evaluation queries:  272 (66.7%) | 428 rows


In [25]:
# verify query overlap is strictly 0
overlap = dev_queries.intersection(eval_queries)
print(f"Query overlap between dev and eval: {len(overlap)} (must be 0)")

# show category distribution in both sets
print("\nTop categories in Development set:")
print(dev_set['area_name'].value_counts().head(5))
print("\nTop categories in Evaluation set:")
print(eval_set['area_name'].value_counts().head(5))

Query overlap between dev and eval: 0 (must be 0)

Top categories in Development set:
area_name
Työturvallisuus                            120
Käsityökalut                                35
Materiaalin mekaaninen työstö               21
Tekniset kemikaalit/-tarvikkeet, teipit     14
Kiinnitystarvikkeet                         13
Name: count, dtype: int64

Top categories in Evaluation set:
area_name
Työturvallisuus                            170
Käsityökalut                                80
Tekniset kemikaalit/-tarvikkeet, teipit     40
Materiaalin mekaaninen työstö               33
Työmaa tarvikkeet ja -varusteet             31
Name: count, dtype: int64


## 5 Export cleaned datasets to data/clean/

In [26]:
export_cols = [
    'roq', 'clean_query', 'request', 'requirements_json',
    'gold_product_number', 'product_matched', 'product_label',
    'area_name', 'group_name', 'class_name'
]
export_cols = [c for c in export_cols if c in matched_rfq.columns]

In [ ]:
# Prepare dataframes
dev_clean = dev_set[export_cols].copy()
eval_clean = eval_set[export_cols].copy()
all_clean = matched_rfq[export_cols].copy()

# Save datasets
out_dir = "../data/clean"
os.makedirs(out_dir, exist_ok=True)
dev_path = os.path.join(out_dir, "test_set_dev.csv")
eval_path = os.path.join(out_dir, "test_set_eval.csv")
all_path = os.path.join(out_dir, "test_set_all.csv")
unlinked_path = os.path.join(out_dir, "unlinked_rfq.csv")

dev_clean.to_csv(dev_path, index=False)
eval_clean.to_csv(eval_path, index=False)
all_clean.to_csv(all_path, index=False)
unlinked_rfq.to_csv(unlinked_path, index=False)

print(f"Saved Development set: {len(dev_clean):,} rows -> '{dev_path}' ({os.path.getsize(dev_path)/1024:.1f} KB)")
print(f"Saved Evaluation set:  {len(eval_clean):,} rows -> '{eval_path}' ({os.path.getsize(eval_path)/1024:.1f} KB)")
print(f"Saved Full test set:   {len(all_clean):,} rows -> '{all_path}' ({os.path.getsize(all_path)/1024:.1f} KB)")
print(f"Saved Unlinked items:  {len(unlinked_rfq):,} rows -> '{unlinked_path}' ({os.path.getsize(unlinked_path)/1024:.1f} KB)")

# Preview first 2 rows of the development set
dev_clean[['roq', 'clean_query', 'gold_product_number', 'product_label']].head(2)

Saved Development set: 235 rows -> '../data/clean/test_set_dev.csv' (209.6 KB)
Saved Evaluation set:  428 rows -> '../data/clean/test_set_eval.csv' (303.6 KB)
Saved Full test set:   663 rows -> '../data/clean/test_set_all.csv' (513.1 KB)
Saved Unlinked items:  29 rows -> '../data/clean/unlinked_rfq.csv' (31.5 KB)


,roq,clean_query,gold_product_number,product_label
0,1,turvakengät ja - nilkkurit kesäkäyttöön turvak...,M418156035,TURVAJ ROCK LOW 35
1,1,turvakengät ja - nilkkurit kesäkäyttöön turvak...,M418156036,TURVAJ ROCK LOW 36
